# Spark X2.6-4B full native multimodal fine-tune — direct one-shot Colab

**Full curriculum, not an image-only subset.** This notebook runs the native Spark multimodal runtime from `colab/final`: image/spatial, real video and time, audiovisual, speech input, general-sound input, native speech output, native sound/music output, joint full tuning, and the two instructor-guided RSI rounds. All configured stages update all Spark parameters; codecs/autoencoders stay frozen infrastructure.

**No CPU preflight. No qualification profile. No Drive. No prior-VM state.** It is deliberately one-shot: clone the reviewed GitHub source, install one CUDA environment, download components and the full `TassP0lar/SparkX2.6-MM` dataset, build the model, and run directly. The old gated launcher remains archived as [`Spark_Formal_Omni_Gated.ipynb`](Spark_Formal_Omni_Gated.ipynb) for history.

**Hardware and access.** One CUDA GPU with **≥90 GB** and native BF16 (G4 / RTX PRO 6000 Blackwell is the target). Enable Colab Secrets `HF_TOKEN` and `OPENCODE_API_KEY`. The HF account must be `TassP0lar`, have read access to the gated components (including Stable Audio Open 1.0), and write access to the private checkpoint and model repositories. The instructor has a **$50** cumulative cap and two 32-example rounds; every request is ledgered.

**Data.** Full dataset `TassP0lar/SparkX2.6-MM` is downloaded from its pinned revision. The catalogue is 164 MB; media are streamed into a bounded 23 GB local TAR cache on demand. No 59 GB LFS clone or Drive mount. The full corpus is 304,861 instructions / 315,359 supervised views / 290,685 training views across ten components.

**One-shot semantics.** `DIRECT_WALL_HOURS` bounds the run so it can stop at an optimizer boundary and upload a complete HF checkpoint. If the curriculum finishes, the final private model is exported and uploaded. If it stops early, the checkpoint repository retains the latest complete snapshot; no qualification or previous-VM profile is read. The full curriculum is large, so a single Colab session may not complete it. Starting a fresh one-shot run does not depend on the prior VM.

## 1. Configuration

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-VL.git"
REPO_REF = "codex/formal-colab"
REPO_COMMIT = ""
WORK = Path("/content/spark-mm-direct")
REPO = WORK / "repo"

RUN_ID = "spark-mm-direct-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime())
DIRECT_WALL_HOURS = 0.0        # 0 = run until complete; set e.g. 11.0 for a bounded session
PROJECTED_HOURS = 72.0         # reporting/budget field only; enforcement is off
HF_CHECKPOINT_REPO = "TassP0lar/SparkX2.6-MM-checkpoints"
HF_MODEL_REPO = "TassP0lar/SparkX2.6-MM"

MIN_GPU_GIB = 90
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps({"run_id": RUN_ID, "work": str(WORK), "wall_hours": DIRECT_WALL_HOURS,
                  "min_gpu_gib": MIN_GPU_GIB}, indent=2))

## 2. GPU contract

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime, then rerun.")
if torch.version.hip:
    raise RuntimeError("This notebook requires a CUDA build, not ROCm.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The assigned GPU does not report native BF16 support.")
props = torch.cuda.get_device_properties(0)
gpu_gib = props.total_memory / 2**30
print(json.dumps({"gpu": props.name, "total_gib": round(gpu_gib, 2),
                  "capability": list(torch.cuda.get_device_capability(0)),
                  "torch": torch.__version__, "cuda": torch.version.cuda}, indent=2))
if gpu_gib < MIN_GPU_GIB:
    raise RuntimeError(f"Full native multimodal training needs >={MIN_GPU_GIB} GiB; found {gpu_gib:.1f} GiB")

## 3. Clone the reviewed source

In [ ]:
def run(command, cwd=None, env=None, check=True):
    print("$", " ".join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], cwd=None if cwd is None else str(cwd),
                          env=env, check=check, text=True)

if REPO.exists():
    shutil.rmtree(REPO)
REPO.parent.mkdir(parents=True, exist_ok=True)
if REPO_COMMIT:
    REPO.mkdir()
    run(["git", "init"], cwd=REPO)
    run(["git", "remote", "add", "origin", REPO_URL], cwd=REPO)
    run(["git", "fetch", "--depth", "1", "origin", REPO_COMMIT], cwd=REPO)
    run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=REPO)
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO])

required = [
    "colab/final/run.py", "colab/final/engine.py", "colab/final/model.py",
    "colab/final/data.py", "colab/final/assets.py", "colab/final/storage.py",
    "colab/final/teacher.py", "colab/final/bootstrap.py",
    "colab/final/campaign.json", "colab/final/requirements.lock",
    "colab/final/bootstrap.lock.json", "colab/final/components.lock.json",
    "colab/final/dataset.lock.json", "colab/formal/HF_FULL_DATASET_HANDOFF.json",
    "colab/formal/teacher_readiness.py", "colab/preflight.py", "colab/stage1/common.py",
]
missing = [path for path in required if not (REPO / path).is_file()]
if missing:
    raise RuntimeError("GitHub checkout lacks the full runtime:\n  " + "\n  ".join(missing))
sys.path.insert(0, str(REPO))
head = run(["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip()
h = hashlib.sha256()
for name in ("colab/final/run.py", "colab/final/engine.py", "colab/final/storage.py",
             "colab/final/teacher.py", "colab/final/campaign.json",
             "colab/final/requirements.lock"):
    h.update((REPO / name).read_bytes())
h.update(head.encode())
SOURCE_SHA = h.hexdigest()
print("SOURCE_COMMIT", head)
print("SOURCE_SHA256", SOURCE_SHA)

## 4. Install the CUDA environment once

In [ ]:
from colab.final.bootstrap import install_environment

PYTHON = Path(install_environment(WORK, REPO, "cuda", base_python=sys.executable))
if not PYTHON.is_file():
    raise RuntimeError("The isolated CUDA environment did not produce a Python executable")
print("PYTHON", PYTHON)

## 5. Secrets and direct campaign configuration

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
INSTRUCTOR_KEY = userdata.get("OPENCODE_API_KEY")
if not HF_TOKEN:
    raise RuntimeError("Enable HF_TOKEN in Colab Secrets")
if not INSTRUCTOR_KEY:
    raise RuntimeError("Enable OPENCODE_API_KEY in Colab Secrets: instructor RSI is part of the full curriculum")

CONFIG = json.loads((REPO / "colab/final/campaign.json").read_text())
CONFIG.update({
    "campaign_id": "spark-mm-direct",
    "work_root": str(WORK),
    "model_repo": HF_MODEL_REPO,
    "model_repo_private": True,
    "checkpoint_backend": "huggingface",
    "checkpoint_repo": HF_CHECKPOINT_REPO,
    "checkpoint_repo_private": True,
    "drive_project_folder": None,          # HF checkpoints + local artifacts only
    "instructor_enabled": True,            # keep both instructor RSI rounds
    "instructor_route_eligibility_receipt": None,
    "enforce_compute_budget": False,       # no CU-balance/rate gate
    "direct_wall_hours": DIRECT_WALL_HOURS,
    "direct_projected_hours": PROJECTED_HOURS,
    "observed_compute_balance": None,
    "observed_cu_per_hour": None,
})
from colab.final.common import validate_config
validate_config(CONFIG)
CONFIG_PATH = WORK / "direct-campaign.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + "\n")
print(json.dumps({"stages": [s["name"] for s in CONFIG["stages"]],
                  "instructor_rounds": CONFIG["instructor_rounds"],
                  "instructor_records_per_round": CONFIG["instructor_records_per_round"],
                  "drive_backend": bool(CONFIG["drive_project_folder"]),
                  "budget_enforced": CONFIG["enforce_compute_budget"]}, indent=2))

## 6. Run the full direct campaign

This downloads the pinned components and builds the full dataset on the GPU VM, then runs bridge alignment, image/video/audio/joint full stages, instructor RSI, evaluation and final export. Events stream directly from the runner.

In [ ]:
env = os.environ.copy()
env.update({
    "PYTHONPATH": str(REPO),
    "HF_HOME": str(WORK / "hf"),
    "TORCH_HOME": str(WORK / "torch"),
    "UNSLOTH_STUDIO_HOME": str(WORK / "studio"),
    "UNSLOTH_COMPILE_LOCATION": str(WORK / "unsloth-compiled"),
    "TORCHINDUCTOR_CACHE_DIR": str(WORK / "torchinductor"),
    "TRITON_CACHE_DIR": str(WORK / "triton"),
    "XDG_CACHE_HOME": str(WORK / "cache"),
    "UV_CACHE_DIR": str(WORK / "uv-cache"),
    "PYTHONDONTWRITEBYTECODE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "WANDB_DISABLED": "true",
    "UNSLOTH_SKIP_AUTOSTART": "1",
    "HF_HUB_DISABLE_XET": "1",
})
secrets = json.dumps({"HF_TOKEN": HF_TOKEN, "OPENCODE_API_KEY": INSTRUCTOR_KEY}) + "\n"
command = [PYTHON, "-m", "colab.final.run", "direct",
           "--config", CONFIG_PATH, "--source-sha", SOURCE_SHA]
RUN = subprocess.run([str(part) for part in command], cwd=REPO, env=env,
                     input=secrets, text=True)
print("DIRECT_EXIT_CODE", RUN.returncode)

## 7. Receipts

In [ ]:
import json
from pathlib import Path

def load(name):
    path = WORK / name
    return json.loads(path.read_text()) if path.exists() else None

campaign = load("CAMPAIGN.json") or {}
environment = load("ENVIRONMENT.json") or {}
profile = load("DIRECT_PROFILE.json") or {}
model_upload = load("MODEL_UPLOAD_RECEIPT.json")
checkpoints = sorted([p for p in (WORK / "checkpoints").glob("*") if p.is_dir()]) if (WORK / "checkpoints").exists() else []
receipt = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "source_sha256": SOURCE_SHA,
    "direct_exit_code": RUN.returncode,
    "environment": environment,
    "profile": profile,
    "training_complete": campaign.get("training_complete"),
    "completed_stages": campaign.get("completed_stages", []),
    "rsi_rounds": campaign.get("rsi_rounds", []),
    "stopped_reason": campaign.get("stopped_reason"),
    "last_checkpoint": campaign.get("last_checkpoint"),
    "local_checkpoints": [str(path) for path in checkpoints],
    "final_model_upload": model_upload,
    "capability_promoted": False,
}
path = WORK / "DIRECT_RUN_RECEIPT.json"
path.write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
print("Receipt:", path)
if RUN.returncode != 0:
    raise RuntimeError("Direct campaign exited nonzero; inspect the streamed event above and the receipt")